# PoC Fase 0 — Caption testuali vs Codice CAD parametrico

**Obiettivo:** per ognuno dei 6 test case, generare codice CAD dall'immagine del pezzo e confrontare
i risultati di ricerca (approccio attuale con caption vs nuovo con codice CAD) rispetto agli expected/unwanted noti.

**Limitazione nota:** l'indice Weaviate contiene ancora caption testuali, non codice CAD.
Il confronto non è pulito: la parte BM25 non funzionerà bene (vocabolari diversi: query con funzioni Python,
documenti con frasi in italiano), ma la parte vettoriale sì perché l'embedding cattura la similarità semantica
indipendentemente dal formato. Il test misura principalmente se l'embedding del codice CAD trova pezzi
semanticamente simili rispetto all'embedding delle caption testuali.

**Go/no-go:** se Expected CAD ≥ Expected caption su almeno 4 dei 6 test case, l'approccio funziona
e si procede con la Fase 1.

| Metrica | Cosa ci dice |
|---|---|
| CAD OK | GPT-4.1 genera codice Python sintatticamente valido |
| t(s) | Latenza generazione — confronto con caption (~2s) |
| Expected caption/CAD | % pezzi attesi trovati nei top-20 |
| Unwanted caption/CAD | % pezzi indesiderati esclusi dai top-20 |

---
## Cella 1 — Installazione dipendenze

In [ ]:
!pip install -q openai weaviate-client google-auth

---
## Cella 2 — Configurazione

La chiave OpenAI viene letta dal secret Colab `OPENAI`.
Il file `sa.json` (service account GCP) viene caricato da Google Drive o upload manuale
e serve per autenticarsi con Vertex AI (il vectorizer usato dal cluster Weaviate).

In [ ]:
import json
import os
from google.colab import userdata, files
from google.oauth2 import service_account
from google.auth.transport.requests import Request

# --- OpenAI ---
OPENAI_API_KEY = userdata.get("OPENAI")

# --- Weaviate ---
_raw_url = userdata.get("WEAVIATE_URL")
WEAVIATE_URL     = _raw_url if _raw_url.startswith("https://") else f"https://{_raw_url}"
WEAVIATE_API_KEY = userdata.get("WEAVIATE_API_KEY")
COLLECTION       = "Sinde4"

print(f"WEAVIATE_URL: {WEAVIATE_URL}")

# --- Service Account GCP (per Vertex AI vectorizer) ---
SA_PATH = "sa.json"
if os.path.exists(SA_PATH):
    print(f"sa.json già presente, lo riutilizzo.")
    with open(SA_PATH, "r") as f:
        sa_info = json.load(f)
else:
    print("Carica il file sa.json (service account GCP):")
    uploaded = files.upload()
    sa_filename = list(uploaded.keys())[0]
    sa_info = json.loads(uploaded[sa_filename].decode("utf-8"))

SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
creds = service_account.Credentials.from_service_account_info(sa_info, scopes=SCOPES)
creds.refresh(Request())

VERTEX_TOKEN = creds.token
VERTEX_PROJECT = sa_info.get("project_id", "")

print(f"Vertex token ottenuto (prefix: {VERTEX_TOKEN[:10]}...)")
print(f"Vertex project: {VERTEX_PROJECT}")

---
## Cella 3 — Funzioni di generazione CAD

Copia integrale delle funzioni da `poc_phase0.py`:
- `_estrai_blocco()` — estrae un blocco di codice dalla risposta LLM
- `valida_sintassi_python()` — verifica che il codice sia Python valido senza eseguirlo
- `genera_cad_e_parametri()` — prompt unificato che produce codice build123d + JSON parametri
- `genera_con_retry()` — wrapper con retry automatico in caso di errore di sintassi

In [ ]:
import ast
import json
import re
from typing import Optional

from openai import OpenAI

_openai_client = OpenAI(api_key=OPENAI_API_KEY)


# ---------------------------------------------------------------------------
# Parsing della risposta LLM
# ---------------------------------------------------------------------------

def _estrai_blocco(testo: str, linguaggio: str) -> Optional[str]:
    """Estrae il primo blocco ```linguaggio ... ``` dalla risposta."""
    pattern = rf"```{linguaggio}\s*\n(.*?)```"
    match = re.search(pattern, testo, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None


def valida_sintassi_python(codice: str) -> Optional[str]:
    """
    Verifica che il codice sia Python sintatticamente valido senza eseguirlo.
    Restituisce None se OK, oppure il messaggio di errore.
    """
    try:
        ast.parse(codice)
        return None
    except SyntaxError as e:
        return f"SyntaxError riga {e.lineno}: {e.msg}"


# ---------------------------------------------------------------------------
# Prompt per la generazione CAD
# ---------------------------------------------------------------------------

PROMPT_SISTEMA = (
    "Sei un esperto di disegno meccanico e modellazione CAD parametrica. "
    "Riceverai immagini di tavole tecniche meccaniche. "
    "Analizza la geometria del pezzo usando tutte le viste disponibili "
    "(frontale, laterale, sezione) e produci due output distinti come richiesto."
)

PROMPT_UTENTE = """Analizza questo disegno tecnico meccanico e produci ESATTAMENTE due output.

## OUTPUT 1 — Codice build123d (Python)

Scrivi una funzione `gen_step()` in build123d che ricrea la geometria del pezzo.

Regole:
- Usa SOLO funzioni build123d standard: Cylinder, Box, Extrude, Revolve, Hole, Fillet, Chamfer, Loft, Sweep, Shell, PolarLocations, GridLocations, BuildPart, BuildSketch, Plane, Circle, Rectangle, Polygon, add, subtract.
- Unità: millimetri.
- Se ci sono più viste (frontale, laterale, sezione), integrali per ricostruire il modello 3D.
- Se una dimensione non è leggibile, stima dalle proporzioni visive.
- Inizia sempre con `from build123d import *`.
- Il codice deve essere sintatticamente valido Python.

## OUTPUT 2 — Parametri strutturati (JSON)

Estrai un JSON con i parametri geometrici chiave:
{
    "part_type": "shaft|plate|housing|bracket|gear|flange|bushing|other",
    "overall_length_mm": <numero>,
    "max_diameter_mm": <numero o null se non cilindrico>,
    "max_width_mm": <numero o null se cilindrico>,
    "max_height_mm": <numero o null>,
    "features": ["bore", "thread", "flange", "step", "groove", "keyway", "fillet", "chamfer", "slot", "pocket", "rib", "hole"],
    "n_bores": <numero intero>,
    "n_steps": <numero di gradini/spalle>,
    "symmetry": "axial|planar|none",
    "is_hollow": true|false
}

Rispondi ESATTAMENTE in questo formato, senza testo aggiuntivo prima o dopo:

```python
<codice build123d qui>
```

```json
<JSON parametri qui>
```"""


# ---------------------------------------------------------------------------
# Generazione CAD + parametri
# ---------------------------------------------------------------------------

def genera_cad_e_parametri(
    image_b64: str,
    model: str = "gpt-4.1",
    errore_precedente: Optional[str] = None,
) -> dict:
    """
    Genera codice build123d + JSON parametri da un'immagine base64 di disegno tecnico.

    Ritorna:
        {
            "code": str | None,          — codice build123d Python
            "params": dict | None,       — parametri strutturati JSON
            "syntax_error": str | None,  — errore di sintassi Python (se presente)
            "raw_response": str,         — risposta grezza del modello
        }
    """
    prompt_utente = PROMPT_UTENTE
    if errore_precedente:
        prompt_utente = (
            f"ATTENZIONE: il tentativo precedente ha prodotto questo errore Python:\n"
            f"{errore_precedente}\n\n"
            "Correggi il codice e riprova.\n\n"
        ) + prompt_utente

    resp = _openai_client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=2500,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_b64}",
                            "detail": "high",
                        },
                    },
                    {"type": "text", "text": prompt_utente},
                ],
            },
        ],
    )

    raw = (resp.choices[0].message.content or "").strip()

    codice = _estrai_blocco(raw, "python")
    params_raw = _estrai_blocco(raw, "json")

    params = None
    if params_raw:
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError as e:
            print(f"  [parse] JSON non valido: {e}")

    syntax_error = (
        valida_sintassi_python(codice)
        if codice
        else "blocco python non trovato nella risposta"
    )

    return {
        "code": codice,
        "params": params,
        "syntax_error": syntax_error,
        "raw_response": raw,
    }


def genera_con_retry(
    image_b64: str,
    model: str = "gpt-4.1",
    max_tentativi: int = 3,
) -> dict:
    """
    Chiama genera_cad_e_parametri con retry automatico in caso di errore di sintassi.
    Al secondo tentativo passa l'errore al modello come feedback.
    """
    risultato = None
    for tentativo in range(1, max_tentativi + 1):
        errore_precedente = (
            risultato["syntax_error"]
            if risultato and risultato.get("syntax_error")
            else None
        )
        if tentativo > 1:
            print(f"  [retry {tentativo}/{max_tentativi}] errore precedente: {errore_precedente}")

        risultato = genera_cad_e_parametri(
            image_b64, model=model, errore_precedente=errore_precedente
        )

        if not risultato["syntax_error"]:
            if tentativo > 1:
                print(f"  [retry] successo al tentativo {tentativo}")
            return risultato

    print(f"  [retry] fallito dopo {max_tentativi} tentativi")
    return risultato


print("Funzioni di generazione CAD caricate.")

---
## Cella 4 — Connessione Weaviate + fetch immagine

Si connette al cluster Weaviate Cloud e definisce una funzione per recuperare
l'immagine base64 e la caption di un pezzo dato il suo `source_pdf`.

In [ ]:
import weaviate
import requests
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery

# Header per i vectorizer del cluster Weaviate
additional_headers = {
    "X-Openai-Api-Key": OPENAI_API_KEY,          # text2vec-openai vectorizer
    "X-Goog-Vertex-Api-Key": VERTEX_TOKEN,        # eventuale Vertex AI
}
if VERTEX_PROJECT:
    additional_headers["X-Goog-User-Project"] = VERTEX_PROJECT

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers=additional_headers,
)
print(f"Connesso a Weaviate: {client.is_ready()}")

In [ ]:
from weaviate.classes.query import Filter

def fetch_piece(piece_id: str) -> dict | None:
    """
    Recupera da Weaviate l'oggetto il cui source_pdf contiene piece_id.
    Usa un filtro testuale (like) invece di BM25 perché source_pdf
    non ha indexSearchable abilitato.
    """
    coll = client.collections.get(COLLECTION)
    resp = coll.query.fetch_objects(
        filters=Filter.by_property("source_pdf").like(f"*{piece_id}*"),
        limit=1,
        return_properties=["source_pdf", "caption", "image_b64"],
    )
    objects = getattr(resp, "objects", [])
    if objects:
        return objects[0].properties
    return None


# Test rapido
_test = fetch_piece("702.0019.00.0")
if _test:
    print(f"Test OK — trovato: {_test['source_pdf']}")
    print(f"  caption (primi 100 char): {_test.get('caption', '')[:100]}...")
    print(f"  image_b64 presente: {bool(_test.get('image_b64'))}")
else:
    print("ATTENZIONE: pezzo di test non trovato. Verifica COLLECTION e credenziali.")

---
## Cella 5 — Ricerca ibrida con testo arbitrario

Funzione generica che esegue una ricerca ibrida (BM25 + vettoriale) su Weaviate
usando una stringa di testo qualsiasi come query. Verrà usata sia con la caption
testuale (approccio attuale) sia con il codice CAD (nuovo approccio).

In [ ]:
def search_hybrid(
    query_text: str,
    alpha: float = 0.2,
    limit: int = 20,
) -> list[dict]:
    """
    Esegue ricerca ibrida su Weaviate usando query_text come stringa.
    BM25 cerca solo su caption (unica property con indexSearchable).
    """
    coll = client.collections.get(COLLECTION)
    resp = coll.query.hybrid(
        query=query_text,
        alpha=alpha,
        query_properties=["caption"],
        limit=limit,
        return_properties=["source_pdf"],
        return_metadata=MetadataQuery(score=True),
    )
    return [
        {
            "source_pdf": o.properties.get("source_pdf", ""),
            "score": getattr(o.metadata, "score", None),
        }
        for o in getattr(resp, "objects", [])
    ]


# Test rapido
_test_results = search_hybrid("albero cilindrico con flangia", alpha=0.2, limit=3)
print(f"Test ricerca ibrida — {len(_test_results)} risultati:")
for r in _test_results:
    print(f"  {r['source_pdf']:30s}  score={r['score']:.4f}")

---
## Cella 6 — Funzione di valutazione risultati

Confronta i risultati della ricerca con gli expected (pezzi che dovrebbero comparire)
e gli unwanted (pezzi che NON dovrebbero comparire) definiti nei test case.

In [ ]:
def valuta_risultati(
    results: list[dict],
    expected: list[str],
    unwanted: list[str],
) -> dict:
    """
    Controlla quanti expected sono nei top-20 e quanti unwanted sono assenti.
    Il matching è fatto su source_pdf (contiene il nome del file PDF, es. "702.0019.00.0.pdf").
    """
    pdfs = [r["source_pdf"] for r in results]

    expected_trovati  = [e for e in expected if any(e in p for p in pdfs)]
    unwanted_trovati  = [u for u in unwanted if any(u in p for p in pdfs)]
    expected_mancanti = [e for e in expected if e not in expected_trovati]

    return {
        "expected_trovati":  expected_trovati,
        "expected_mancanti": expected_mancanti,
        "unwanted_trovati":  unwanted_trovati,
        "score_expected":    len(expected_trovati) / max(len(expected), 1),
        "score_unwanted":    1 - len(unwanted_trovati) / max(len(unwanted), 1),
    }


print("Funzione di valutazione caricata.")

---
## Cella 7 — Loop sui test case

Per ogni test case:
1. Recupera l'immagine e la caption dal database
2. Esegue la ricerca con la caption attuale (baseline)
3. Genera il codice CAD dall'immagine (con retry)
4. Esegue la ricerca con il codice CAD
5. Confronta i risultati

In [ ]:
import time

TEST_CASES = [
    {"input": "702.0019.00.0",  "expected": ["702.0144.00"],                              "unwanted": ["SM02013561A", "PF00011631-B"]},
    {"input": "PF00009371",     "expected": ["PF00018845", "P00005877A", "PF00015175"],    "unwanted": []},
    {"input": "PF00018620",     "expected": ["704.0256.00.0"],                             "unwanted": ["702.0124.00.0", "702.0144.00.0", "701.1829.00.0"]},
    {"input": "701.2425.00.0",  "expected": ["701.2320.00.0", "701.2332.00.0"],            "unwanted": ["PF00004573", "PF00017257", "701.2146.00.0"]},
    {"input": "PF00016814",     "expected": ["PF00013561", "PF00017176-A"],                "unwanted": ["PF00001092-E"]},
    {"input": "701.1828.00.0",  "expected": ["701.1829.00.0", "701.1831.00.0"],            "unwanted": ["701.2146.00.0", "702.0144.00.0", "PF00002831-A", "PF00002392"]},
]

risultati_globali = []

for tc in TEST_CASES:
    nome = tc["input"]
    print(f"\n{'='*60}")
    print(f"  TEST CASE: {nome}")
    print(f"{'='*60}")

    # 1. Fetch immagine + caption attuale da Weaviate
    piece = fetch_piece(nome)
    if not piece:
        print(f"  [SKIP] pezzo non trovato in Weaviate")
        risultati_globali.append({
            "input": nome, "cad_ok": False, "t_gen_s": 0,
            "params": None, "caption": {}, "cad": {}, "skipped": True,
        })
        continue

    image_b64 = piece.get("image_b64")
    caption   = piece.get("caption", "")

    if not image_b64:
        print(f"  [SKIP] immagine non presente nel database")
        risultati_globali.append({
            "input": nome, "cad_ok": False, "t_gen_s": 0,
            "params": None, "caption": {}, "cad": {}, "skipped": True,
        })
        continue

    print(f"  Caption attuale: {caption[:120]}...")

    # 2. Ricerca ATTUALE con caption (baseline)
    print(f"  Ricerca con caption (alpha=0.2)...")
    res_caption = search_hybrid(caption, alpha=0.2)
    val_caption = valuta_risultati(res_caption, tc["expected"], tc["unwanted"])
    print(f"    Expected trovati: {val_caption['expected_trovati']}")
    print(f"    Expected mancanti: {val_caption['expected_mancanti']}")
    print(f"    Unwanted trovati: {val_caption['unwanted_trovati']}")

    # 3. Genera codice CAD dall'immagine
    print(f"  Generazione codice CAD (GPT-4.1, max 3 tentativi)...")
    t0 = time.time()
    cad = genera_con_retry(image_b64)
    t_gen = time.time() - t0

    cad_ok = cad["syntax_error"] is None
    print(f"    Tempo: {t_gen:.1f}s")
    print(f"    Sintassi OK: {cad_ok}")
    if not cad_ok:
        print(f"    Errore: {cad['syntax_error']}")
    if cad["params"]:
        p = cad["params"]
        print(f"    part_type={p.get('part_type')}  length={p.get('overall_length_mm')}mm  "
              f"diam={p.get('max_diameter_mm')}mm  features={p.get('features')}")
    if cad["code"]:
        n_lines = len(cad["code"].splitlines())
        print(f"    Codice: {n_lines} righe")

    # 4. Ricerca con codice CAD (alpha=0.3: più peso al vettoriale
    #    perché BM25 non matcherà bene codice Python vs caption italiana)
    res_cad = []
    val_cad = {
        "expected_trovati": [], "expected_mancanti": tc["expected"],
        "unwanted_trovati": [], "score_expected": 0, "score_unwanted": 1,
    }
    if cad["code"]:
        print(f"  Ricerca con codice CAD (alpha=0.3)...")
        res_cad = search_hybrid(cad["code"], alpha=0.3)
        val_cad = valuta_risultati(res_cad, tc["expected"], tc["unwanted"])
        print(f"    Expected trovati: {val_cad['expected_trovati']}")
        print(f"    Expected mancanti: {val_cad['expected_mancanti']}")
        print(f"    Unwanted trovati: {val_cad['unwanted_trovati']}")

    risultati_globali.append({
        "input":       nome,
        "cad_ok":      cad_ok,
        "t_gen_s":     round(t_gen, 1),
        "params":      cad["params"],
        "code":        cad["code"],
        "caption":     val_caption,
        "cad":         val_cad,
        "skipped":     False,
    })

print(f"\n\n{'='*60}")
print(f"  COMPLETATO: {len(risultati_globali)} test case elaborati")
print(f"{'='*60}")

---
## Cella 8 — Tabella riassuntiva

Confronto side-by-side dei risultati: caption (approccio attuale) vs codice CAD (nuovo approccio).

In [ ]:
# Intestazione tabella
print(f"\n{'Pezzo':<20} {'CAD OK':^7} {'t(s)':^6}  "
      f"{'Exp caption':^14} {'Exp CAD':^14}  "
      f"{'Unw caption':^14} {'Unw CAD':^14}")
print("-" * 95)

# Contatori per il verdetto finale
cad_wins = 0
caption_wins = 0
ties = 0
cad_ok_count = 0

for r in risultati_globali:
    if r.get("skipped"):
        print(f"{r['input']:<20} {'SKIP':^7} {'—':^6}  {'—':^14} {'—':^14}  {'—':^14} {'—':^14}")
        continue

    c = r["caption"]
    k = r["cad"]

    if r["cad_ok"]:
        cad_ok_count += 1

    # Confronto: il CAD vince se trova più expected O se trova meno unwanted
    cad_score = k.get("score_expected", 0) + k.get("score_unwanted", 0)
    cap_score = c.get("score_expected", 0) + c.get("score_unwanted", 0)
    if cad_score > cap_score:
        cad_wins += 1
        verdict = "CAD+"
    elif cad_score < cap_score:
        caption_wins += 1
        verdict = "CAP+"
    else:
        ties += 1
        verdict = "PARI"

    print(
        f"{r['input']:<20} "
        f"{'SI' if r['cad_ok'] else 'NO':^7} "
        f"{r['t_gen_s']:^6.1f}  "
        f"{c.get('score_expected',0):>5.0%} ({len(c.get('expected_trovati',[]))}/{len(c.get('expected_trovati',[]))+len(c.get('expected_mancanti',[]))})  "
        f"{k.get('score_expected',0):>5.0%} ({len(k.get('expected_trovati',[]))}/{len(k.get('expected_trovati',[]))+len(k.get('expected_mancanti',[]))})  "
        f"{c.get('score_unwanted',0):>5.0%} ({len(c.get('unwanted_trovati',[]))} bad)  "
        f"{k.get('score_unwanted',0):>5.0%} ({len(k.get('unwanted_trovati',[]))} bad)  "
        f"  {verdict}"
    )

# Verdetto finale
total = cad_wins + caption_wins + ties
print(f"\n{'='*95}")
print(f"VERDETTO FINALE")
print(f"{'='*95}")
print(f"  Codice CAD valido:   {cad_ok_count}/{len(risultati_globali)} test case")
print(f"  CAD vince:           {cad_wins}/{total}")
print(f"  Caption vince:       {caption_wins}/{total}")
print(f"  Parità:              {ties}/{total}")
print()

go = cad_wins >= 4  # criterio: CAD vince su almeno 4/6 test case
if go:
    print(f"  >>> GO — CAD vince su {cad_wins}/6 test case (soglia: 4/6). Procedere con Fase 1.")
else:
    print(f"  >>> NO-GO — CAD vince solo su {cad_wins}/6 test case (soglia: 4/6). Serve analisi.")
    print(f"      Nota: il confronto è penalizzato dal BM25 (codice Python vs caption italiana).")
    print(f"      Valutare se con alpha più alto (più vettoriale) il risultato migliora.")

---
## Cella 9 — Dettaglio: codice CAD generato e parametri

Per ogni test case, mostra il codice build123d generato e i parametri strutturati estratti.

In [ ]:
for r in risultati_globali:
    if r.get("skipped"):
        continue

    print(f"\n{'='*70}")
    print(f"  {r['input']}")
    print(f"{'='*70}")

    if r.get("params"):
        print(f"\n--- Parametri strutturati ---")
        print(json.dumps(r["params"], indent=2, ensure_ascii=False))

    if r.get("code"):
        print(f"\n--- Codice build123d ({len(r['code'].splitlines())} righe) ---")
        print(r["code"])
    else:
        print("\n--- Codice CAD non generato ---")

---
## Cella 10 — Esperimento bonus: alpha sweep

Il test principale usa alpha=0.3 per il codice CAD (30% vettoriale, 70% BM25).
Siccome BM25 è penalizzato (codice Python vs caption italiana), proviamo diversi valori
di alpha per capire se il codice CAD funziona meglio con più peso vettoriale.

In [ ]:
alphas_da_testare = [0.2, 0.4, 0.6, 0.8, 1.0]

print(f"{'Alpha':>6}  ", end="")
for tc in TEST_CASES:
    print(f"{tc['input']:<18}", end="")
print(f"  {'Media':>6}")
print("-" * (8 + 18 * len(TEST_CASES) + 8))

for alpha in alphas_da_testare:
    scores = []
    print(f"{alpha:>6.1f}  ", end="")

    for i, r in enumerate(risultati_globali):
        if r.get("skipped") or not r.get("code"):
            print(f"{'—':^18}", end="")
            continue

        tc = TEST_CASES[i]
        res = search_hybrid(r["code"], alpha=alpha)
        val = valuta_risultati(res, tc["expected"], tc["unwanted"])
        s = val["score_expected"]
        scores.append(s)
        print(f"{s:>5.0%} ({len(val['expected_trovati'])}/{len(tc['expected'])}){' ':>8}", end="")

    avg = sum(scores) / max(len(scores), 1)
    print(f"  {avg:>5.0%}")

print("\nAlpha: 0.0 = solo BM25, 1.0 = solo vettoriale")
print("Se i risultati migliorano con alpha alto, l'embedding del codice CAD funziona.")

---
# Fase 1 — Ingestion CAD-su-CAD (subset 200 pezzi)

L'obiettivo è creare una nuova collection `SindeCAD` dove ogni documento contiene:
- **cad_code** (testo embeddato) — il codice build123d generato da GPT-4.1
- **params_json** — parametri strutturati JSON
- **source_pdf** — nome del file PDF originale
- **image_b64** — immagine originale del disegno

La ricerca sarà CAD-su-CAD: query con codice CAD, documenti con codice CAD.

**Prerequisiti:**
- La cartella `SubCases` deve essere accessibile su Google Drive
- Contiene ~200 PDF di disegni tecnici

---
## Cella 11 — Cleanup

Chiudi la connessione Weaviate.

In [ ]:
client.close()
print("Connessione Weaviate chiusa.")

## Cella 12 — Mount Google Drive e lista PDF

In [ ]:
!pip install -q pypdfium2 Pillow

from google.colab import drive
import glob

drive.mount("/content/drive")

# Percorso della cartella SubCases su Drive — MODIFICA se diverso
SUBCASES_PATH = "/content/drive/MyDrive/SubCases"

pdf_files = sorted(glob.glob(f"{SUBCASES_PATH}/*.pdf", recursive=False))
# Includi anche PDF in sottocartelle se presenti
if not pdf_files:
    pdf_files = sorted(glob.glob(f"{SUBCASES_PATH}/**/*.pdf", recursive=True))

print(f"Trovati {len(pdf_files)} PDF in {SUBCASES_PATH}")
if pdf_files:
    print(f"  Primi 5: {[os.path.basename(f) for f in pdf_files[:5]]}")
    print(f"  Ultimi 5: {[os.path.basename(f) for f in pdf_files[-5:]]}")
else:
    print("ATTENZIONE: nessun PDF trovato. Verifica il percorso SUBCASES_PATH.")

## Cella 13 — Creazione collection SindeCAD

Crea la nuova collection con `text2vec-openai` come vectorizer.
Solo `cad_code` viene embeddato — le altre property non contribuiscono al vettore.

In [ ]:
from weaviate.classes.config import Configure, Property, DataType, Tokenization

CAD_COLLECTION = "SindeCAD"

# Elimina la collection se esiste già (per poter ricreare)
if client.collections.exists(CAD_COLLECTION):
    print(f"Collection {CAD_COLLECTION} già esistente — la elimino per ricrearla pulita.")
    client.collections.delete(CAD_COLLECTION)

client.collections.create(
    name=CAD_COLLECTION,
    vectorizer_config=Configure.Vectorizer.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        Property(
            name="cad_code",
            data_type=DataType.TEXT,
            description="Codice build123d Python generato dal disegno tecnico",
            skip_vectorization=False,       # QUESTO viene embeddato
            tokenization=Tokenization.WORD,  # BM25 abilitato
        ),
        Property(
            name="params_json",
            data_type=DataType.TEXT,
            description="Parametri strutturati JSON (part_type, dimensioni, features)",
            skip_vectorization=True,
            tokenization=Tokenization.WORD,
        ),
        Property(
            name="source_pdf",
            data_type=DataType.TEXT,
            description="Nome del file PDF originale",
            skip_vectorization=True,
            tokenization=Tokenization.FIELD,
        ),
        Property(
            name="image_b64",
            data_type=DataType.BLOB,
            description="Immagine PNG base64 del disegno tecnico",
        ),
    ],
)

print(f"Collection {CAD_COLLECTION} creata con successo.")
cad_coll = client.collections.get(CAD_COLLECTION)
print(f"  Properties: {[p.name for p in cad_coll.config.get().properties]}")

## Cella 14 — Ingestion: PDF → CAD → Weaviate

Per ogni PDF:
1. Converte la prima pagina in PNG base64
2. Genera codice CAD + parametri con GPT-4.1 (con retry)
3. Inserisce in SindeCAD

Stima tempo: ~8s/pezzo × 200 pezzi ≈ 27 minuti.
Lo stato viene salvato su disco (`ingestion_state.json`) per poter riprendere in caso di interruzione.

In [ ]:
import base64
from io import BytesIO
import pypdfium2 as pdfium

def pdf_to_b64(pdf_path: str) -> str | None:
    """Converte la prima pagina di un PDF in PNG base64."""
    try:
        data = open(pdf_path, "rb").read()
        pdf = pdfium.PdfDocument(data)
        if len(pdf) == 0:
            return None
        page = pdf[0]
        bitmap = page.render(scale=3)
        pil_img = bitmap.to_pil()
        buf = BytesIO()
        pil_img.save(buf, format="PNG")
        return base64.b64encode(buf.getvalue()).decode("utf-8")
    except Exception as e:
        print(f"  [pdf] errore: {e}")
        return None


# --- Stato per resume ---
STATE_FILE = "ingestion_state.json"

def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    return {"done": [], "failed": []}

def save_state(state: dict):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)


# --- Ingestion loop ---
cad_coll = client.collections.get(CAD_COLLECTION)
state = load_state()
done_set = set(state["done"])

to_process = [f for f in pdf_files if os.path.basename(f) not in done_set]
print(f"Già processati: {len(done_set)}/{len(pdf_files)}")
print(f"Da processare: {len(to_process)}")
print()

ok_count = 0
fail_count = 0

for i, pdf_path in enumerate(to_process):
    fname = os.path.basename(pdf_path)
    print(f"[{len(done_set)+i+1}/{len(pdf_files)}] {fname}...", end=" ", flush=True)

    # 1. PDF → PNG base64
    img_b64 = pdf_to_b64(pdf_path)
    if not img_b64:
        print("SKIP (PDF non convertibile)")
        state["failed"].append({"file": fname, "reason": "pdf_conversion"})
        save_state(state)
        fail_count += 1
        continue

    # 2. Genera codice CAD
    t0 = time.time()
    try:
        cad = genera_con_retry(img_b64, max_tentativi=3)
    except Exception as e:
        print(f"ERRORE GPT ({e})")
        state["failed"].append({"file": fname, "reason": str(e)})
        save_state(state)
        fail_count += 1
        continue
    t_gen = time.time() - t0

    if not cad["code"] or cad["syntax_error"]:
        print(f"SKIP (sintassi invalida: {cad.get('syntax_error', 'no code')}) [{t_gen:.1f}s]")
        state["failed"].append({"file": fname, "reason": f"syntax: {cad.get('syntax_error')}"})
        save_state(state)
        fail_count += 1
        continue

    # 3. Insert in SindeCAD
    try:
        cad_coll.data.insert({
            "cad_code": cad["code"],
            "params_json": json.dumps(cad["params"]) if cad["params"] else "{}",
            "source_pdf": fname,
            "image_b64": img_b64,
        })
    except Exception as e:
        print(f"ERRORE Weaviate ({e}) [{t_gen:.1f}s]")
        state["failed"].append({"file": fname, "reason": f"weaviate: {e}"})
        save_state(state)
        fail_count += 1
        continue

    print(f"OK [{t_gen:.1f}s] {cad['params'].get('part_type', '?') if cad['params'] else '?'}")
    state["done"].append(fname)
    save_state(state)
    ok_count += 1

print(f"\n{'='*60}")
print(f"  Ingestion completata")
print(f"  OK: {ok_count}  |  Falliti: {fail_count}  |  Totale collection: {len(state['done'])}")
print(f"{'='*60}")

## Cella 15 — Test CAD-su-CAD con i 6 test case

Ora cerchiamo su `SindeCAD` usando il codice CAD come query.
Il confronto è finalmente equo: codice CAD cerca in un indice di codice CAD.

In [ ]:
def search_cad_hybrid(
    query_text: str,
    alpha: float = 0.5,
    limit: int = 20,
) -> list[dict]:
    """Ricerca ibrida su SindeCAD (codice CAD su codice CAD)."""
    cad_coll = client.collections.get(CAD_COLLECTION)
    resp = cad_coll.query.hybrid(
        query=query_text,
        alpha=alpha,
        query_properties=["cad_code"],
        limit=limit,
        return_properties=["source_pdf"],
        return_metadata=MetadataQuery(score=True),
    )
    return [
        {
            "source_pdf": o.properties.get("source_pdf", ""),
            "score": getattr(o.metadata, "score", None),
        }
        for o in getattr(resp, "objects", [])
    ]


# --- Loop test case su SindeCAD ---
print("="*80)
print("  TEST CAD-su-CAD (collection SindeCAD)")
print("="*80)

risultati_cad_cad = []

for tc, r_old in zip(TEST_CASES, risultati_globali):
    nome = tc["input"]
    print(f"\n--- {nome} ---")

    if r_old.get("skipped") or not r_old.get("code"):
        print("  [SKIP] nessun codice CAD disponibile dalla Fase 0")
        risultati_cad_cad.append(None)
        continue

    cad_code = r_old["code"]

    # Ricerca su SindeCAD
    res = search_cad_hybrid(cad_code, alpha=0.5)
    val = valuta_risultati(res, tc["expected"], tc["unwanted"])

    print(f"  Expected trovati:  {val['expected_trovati']}")
    print(f"  Expected mancanti: {val['expected_mancanti']}")
    print(f"  Unwanted trovati:  {val['unwanted_trovati']}")
    print(f"  Top-5 risultati:")
    for j, r in enumerate(res[:5]):
        print(f"    {j+1}. {r['source_pdf']:<30s}  score={r['score']:.4f}")

    risultati_cad_cad.append(val)


# --- Tabella comparativa ---
print(f"\n\n{'='*100}")
print(f"  CONFRONTO: Caption (Sinde4) vs CAD-su-CAD (SindeCAD)")
print(f"{'='*100}")
print(f"{'Pezzo':<20} {'Exp Caption':^14} {'Exp CadSuCad':^14} {'Unw Caption':^14} {'Unw CadSuCad':^14} {'Vincitore':^10}")
print("-" * 100)

cad_wins_f1 = 0
cap_wins_f1 = 0
ties_f1 = 0

for i, tc in enumerate(TEST_CASES):
    r_old = risultati_globali[i]
    r_new = risultati_cad_cad[i]

    if r_old.get("skipped") or r_new is None:
        print(f"{tc['input']:<20} {'—':^14} {'—':^14} {'—':^14} {'—':^14} {'SKIP':^10}")
        continue

    c = r_old["caption"]
    k = r_new

    cap_score = c.get("score_expected", 0) + c.get("score_unwanted", 0)
    cad_score = k.get("score_expected", 0) + k.get("score_unwanted", 0)

    if cad_score > cap_score:
        cad_wins_f1 += 1
        v = "CAD+"
    elif cad_score < cap_score:
        cap_wins_f1 += 1
        v = "CAP+"
    else:
        ties_f1 += 1
        v = "PARI"

    n_exp = len(tc["expected"])
    n_unw = len(tc["unwanted"])

    print(
        f"{tc['input']:<20} "
        f"{c.get('score_expected',0):>5.0%} ({len(c.get('expected_trovati',[]))}/{n_exp})  "
        f"{k.get('score_expected',0):>5.0%} ({len(k.get('expected_trovati',[]))}/{n_exp})  "
        f"{c.get('score_unwanted',0):>5.0%} ({len(c.get('unwanted_trovati',[]))} bad)  "
        f"{k.get('score_unwanted',0):>5.0%} ({len(k.get('unwanted_trovati',[]))} bad)  "
        f"  {v:^10}"
    )

total_f1 = cad_wins_f1 + cap_wins_f1 + ties_f1
print(f"\n{'='*100}")
print(f"  CAD-su-CAD vince: {cad_wins_f1}/{total_f1}")
print(f"  Caption vince:    {cap_wins_f1}/{total_f1}")
print(f"  Parità:           {ties_f1}/{total_f1}")

go_f1 = cad_wins_f1 >= 4
if go_f1:
    print(f"\n  >>> GO — CAD-su-CAD vince su {cad_wins_f1}/6 test case! Procedere con ingestion completa.")
else:
    print(f"\n  >>> Risultato: CAD-su-CAD vince su {cad_wins_f1}/6 test case.")